# Pharos — swimmer detector fine-tune

Fine-tunes **YOLO11-nano** on swimmer bounding boxes so Pharos can see people **in** the water (stock MoveNet can't). Trains on a free Colab T4 GPU in ~2–4 hours, exports an ONNX model to run in the browser via onnxruntime-web.

**Before you start — 3 things only you can do:**
1. **Runtime → Change runtime type → T4 GPU** (the free GPU is not on by default).
2. A free **Roboflow** account for the dataset (Settings → API key). *Or* use the upload-a-zip cell instead — no signup.
3. Don't let the tab sit idle >90 min mid-training, or the free runtime disconnects. It checkpoints every epoch to Drive, so a disconnect isn't fatal.

**Never** put the Pharos eval clips (`floats/lanepool/nightpool`) or any real-rescue footage into training data. Those are the held-out test set.

In [ ]:
# 1. Confirm the GPU is on. If this prints 'no GPU', fix Runtime → Change runtime type → T4 GPU first.
import subprocess
out = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(out.stdout if out.returncode == 0 else 'no GPU — set Runtime > Change runtime type > T4 GPU, then re-run this cell')

In [ ]:
# 2. Install Ultralytics (YOLO) + Roboflow. ~1 min.
%pip install -q ultralytics roboflow
import ultralytics; ultralytics.checks()

In [ ]:
# 3. (Recommended) Mount Google Drive so checkpoints survive a disconnect.
#    One OAuth click. Skip this cell if you'd rather not — then training saves only to the ephemeral runtime.
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/pharos-finetune'
import os; os.makedirs(PROJECT_DIR, exist_ok=True)
print('checkpoints →', PROJECT_DIR)

## 4. Get the dataset — pick ONE path

**Path A (Roboflow, recommended):** free account → Settings → copy your API key. Pulls the two verified pool datasets. Run cell 4A.

**Path B (no signup):** download a dataset zip yourself, upload it to Colab (folder icon on the left → upload), and run cell 4B instead.

In [ ]:
# 4A. Roboflow path. Paste your free API key when prompted.
#     Pulls ecl/swimmers-detection (Public Domain) + swimmingdetection/swimmer-detection (CC BY 4.0),
#     both above-water pool sets, ~2,000 images total, in YOLO format.
from roboflow import Roboflow
import getpass
RF_KEY = getpass.getpass('Roboflow API key: ')
rf = Roboflow(api_key=RF_KEY)

# Downloads the latest version of each project as YOLOv11 format.
d1 = rf.workspace('ecl-4wc4j').project('swimmers-detection').version(1).download('yolov11', location='/content/ds_ecl')
d2 = rf.workspace('swimmingdetection').project('swimmer-detection-gjwyk').version(1).download('yolov11', location='/content/ds_sd')
print('downloaded to', d1.location, 'and', d2.location)
print('NOTE: if a version(1) call errors, open the dataset page on roboflow.com and use its real version number / export snippet.')

In [ ]:
# 4B. (Alternative, no signup) If you uploaded a dataset zip instead, unzip it here.
#     Edit ZIP to your uploaded filename. It must be YOLO format (images/ + labels/ + a data.yaml).
# import zipfile, os
# ZIP = '/content/your-dataset.zip'
# with zipfile.ZipFile(ZIP) as z: z.extractall('/content/ds_manual')
# print(os.listdir('/content/ds_manual'))

In [ ]:
# 5. Merge the datasets into one 'swimmer' class and write a data.yaml.
#    The source sets use classes like bodysurface / bodyunder / black-hat / umpire.
#    For 'find any person in the water' we collapse the swimmer-in-water classes to a single class 0.
#    Poolside-only classes (umpire, hats worn on deck) are dropped. Adjust KEEP below if you disagree.
import glob, os, shutil, yaml

# class names that mean 'a person in/at the water' — map all of these to our single class 0.
KEEP = {'bodysurface', 'bodyunder', 'Relax', 'human', 'person in water', 'swimmer', 'Swimming'}

MERGED = '/content/pharos_ds'
for sub in ['images/train','images/val','labels/train','labels/val']:
    os.makedirs(f'{MERGED}/{sub}', exist_ok=True)

def ingest(root):
    # find the dataset's own data.yaml to read its class names
    yml = glob.glob(f'{root}/**/data.yaml', recursive=True)
    if not yml: print('no data.yaml in', root); return
    names = yaml.safe_load(open(yml[0]))['names']
    names = {i:n for i,n in (names.items() if isinstance(names, dict) else enumerate(names))}
    for split in ['train','valid','val']:
        for img in glob.glob(f'{os.path.dirname(yml[0])}/{split}/images/*'):
            base = os.path.splitext(os.path.basename(img))[0]
            lbl = f'{os.path.dirname(yml[0])}/{split}/labels/{base}.txt'
            dst = 'val' if split in ('valid','val') else 'train'
            keep_lines = []
            if os.path.exists(lbl):
                for line in open(lbl):
                    p = line.split()
                    if not p: continue
                    cname = names.get(int(p[0]), '')
                    if cname in KEEP:
                        keep_lines.append('0 ' + ' '.join(p[1:]))
            if keep_lines:  # only keep images that actually contain a swimmer
                shutil.copy(img, f'{MERGED}/images/{dst}/{base}.jpg')
                open(f'{MERGED}/labels/{dst}/{base}.txt','w').write('\n'.join(keep_lines))

for root in ['/content/ds_ecl','/content/ds_sd','/content/ds_manual']:
    if os.path.exists(root): ingest(root)

yaml.safe_dump({'path': MERGED, 'train':'images/train', 'val':'images/val',
                'nc':1, 'names':['swimmer']}, open(f'{MERGED}/data.yaml','w'))
print('train imgs:', len(glob.glob(f'{MERGED}/images/train/*')),
      '| val imgs:', len(glob.glob(f'{MERGED}/images/val/*')))

In [ ]:
# 6. Train YOLO11-nano. ~2–4h on a free T4. Watch the mAP50 column climb; stop early if it plateaus.
from ultralytics import YOLO
model = YOLO('yolo11n.pt')  # nano — small enough for a phone
model.train(
    data='/content/pharos_ds/data.yaml',
    imgsz=640, epochs=100, batch=16, patience=20,
    project=PROJECT_DIR if 'PROJECT_DIR' in dir() else '/content/runs',
    name='pharos-swimmer',
)

In [ ]:
# 7. Export the best weights to ONNX (opset 12) for onnxruntime-web in the browser.
best = model.trainer.best  # path to best.pt
print('best weights:', best)
onnx_path = YOLO(best).export(format='onnx', opset=12, imgsz=640)
print('ONNX →', onnx_path)

In [ ]:
# 8. Download the model to your laptop.
from google.colab import files
import shutil
shutil.copy(onnx_path, '/content/pharos-swimmer.onnx')
shutil.copy(str(best), '/content/pharos-swimmer.pt')  # keep the torch weights too, for later re-export
files.download('/content/pharos-swimmer.onnx')
files.download('/content/pharos-swimmer.pt')
print('done — put pharos-swimmer.onnx into the Pharos app and run benchmarks/detect-bench.html before/after.')

## Next: measure it (the whole point)

Back in Pharos, follow `research/finetune-eval-spec.md`:
1. Wire `pharos-swimmer.onnx` into the app via onnxruntime-web (a new detector in `js/detector.js`).
2. Hand-label ~25–30 anchor frames across the three eval clips for In-Water Swimmer Recall.
3. Run stock MoveNet vs the fine-tune through the frozen `benchmarks/detect-bench.html`.
4. Report per clip, with confidence intervals. A clean **0% → X%** is publishable — nobody publishes in-water recall across pools at all.